# 04 · Encapsulation

**Goal:** understand access control conventions (public/protected/private), and the
Pythonic way to add getters/setters using `@property`.

### What is encapsulation?

Encapsulation means **bundling data and the methods that operate on it together**, and
**restricting direct access** to some of an object's internals — so the object controls how
its data is read or changed, rather than letting outside code mess with it directly.

Python doesn't have *true* private variables like Java (there's no compiler enforcement), but
it has strong **naming conventions** that everyone follows.

### The three levels

| Prefix | Convention | Meaning |
|---|---|---|
| `name` | Public | Accessible from anywhere, no restriction |
| `_name` | Protected | "Internal use" — accessible, but signals *please don't touch from outside* |
| `__name` | Private | Name-mangled — hard (not impossible) to access from outside the class |

In [1]:
class Employee:
    def __init__(self, name, salary):
        self.name = name          # public
        self._department = "N/A"  # protected (by convention)
        self.__salary = salary    # private (name-mangled)

e = Employee("Meera", 50000)

print(e.name)          # fine, it's public
print(e._department)   # works, but you're "not supposed to" touch it from outside

try:
    print(e.__salary)  # AttributeError! Python renames __salary internally
except AttributeError as err:
    print("Error:", err)

Meera
N/A
Error: 'Employee' object has no attribute '__salary'


### How "private" actually works: name mangling

Python renames `__salary` to `_Employee__salary` internally. This is NOT true security — it's
meant to avoid accidental clashes in inheritance, not to make data unreadable. You *can* still
access it if you really try, but you shouldn't.

In [2]:
print(e._Employee__salary)   # works, but this is a strong signal you're doing something wrong

50000


### The Pythonic way: `@property` (getters and setters)

Instead of writing `get_salary()` / `set_salary()` methods like in Java, Python lets you use
the `@property` decorator so accessing a "computed" or "controlled" attribute still looks like
plain attribute access (`obj.salary`), while actually running your validation code underneath.

In [3]:
class Employee:
    def __init__(self, name, salary):
        self.name = name
        self.__salary = salary   # store the real value privately

    @property
    def salary(self):
        """Getter: runs when you do `employee.salary`"""
        return self.__salary

    @salary.setter
    def salary(self, value):
        """Setter: runs when you do `employee.salary = value`"""
        if value < 0:
            raise ValueError("Salary cannot be negative")
        self.__salary = value

e = Employee("Meera", 50000)

print(e.salary)      # calls the getter -> 50000, looks like plain attribute access

e.salary = 60000     # calls the setter -> validated and updated
print(e.salary)

try:
    e.salary = -100  # setter rejects this
except ValueError as err:
    print("Error:", err)

50000
60000
Error: Salary cannot be negative


### Read-only attributes

If you define only a getter (no `@x.setter`), the attribute becomes read-only from outside.

In [4]:
class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    def area(self):
        return 3.14159 * self.radius ** 2   # computed on the fly, not stored

c = Circle(5)
print(c.area)

try:
    c.area = 100   # AttributeError: no setter defined
except AttributeError as err:
    print("Error:", err)

78.53975
Error: property 'area' of 'Circle' object has no setter


### Why bother? A before/after comparison

**Without encapsulation**, any code anywhere can silently corrupt your object's state:
```python
account.balance = -99999   # nothing stops this
```

**With encapsulation**, invalid states are impossible because the setter enforces the rules
every single time the attribute is changed, no matter where in the codebase it happens.

### 🧠 Quick check

1. What does a single leading underscore (`_name`) signal, versus a double leading
   underscore (`__name`)?
2. What does `@property` let you do that a plain attribute can't?
3. How do you make an attribute read-only using `@property`?

<details>
<summary>Answers</summary>

1. `_name` = "internal use, please treat as protected" (convention only).
   `__name` = name-mangled to `_ClassName__name`, harder to access accidentally, closer to
   "private" but still not truly enforced.
2. It lets you run validation/logic when the attribute is read or written, while callers still
   use plain `obj.attr` syntax.
3. Define only the `@property` getter, and don't define a matching `@attr.setter`.
</details>

### ✍️ Practice

1. Create a `Temperature` class storing `__celsius` privately.
2. Add a `celsius` property with a setter that rejects values below `-273.15` (absolute zero).
3. Add a **read-only** `fahrenheit` property computed from `__celsius`.

Continue to **`05_inheritance.ipynb`** next.